<a href="https://colab.research.google.com/github/fuadfach/geog761lab/blob/main/Lab3_Exe6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Our usual set up routine
!pip install geemap --quiet

import geemap
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.cluster import KMeans

import ee
ee.Authenticate()
ee.Initialize(project='geog761-ffac001') #<- Remember to change this to your own project's name!

In [ ]:
# Additional required packages
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import urllib.request
import tempfile
import matplotlib.image as mpimg

In [ ]:
# First, lets make our cloud clearing 'worse' in order to more clearly demonstrate our clustering alg
# Tweaked control variables
AOI = ee.Geometry.Point(174.7633, -36.8485)
START_DATE = '2023-01-01'
END_DATE = '2023-03-31'
CLOUD_FILTER = 30
CLD_PRB_THRESH = 30
NIR_DRK_THRESH = 0.15
CLD_PRJ_DIST = 1
BUFFER = 5

### `get_s2_sr_cld_col(aoi, start_date, end_date)` buat join antara dataset Sentinel-2: SR dengan Sentinel-2: Cloud Probability
menjadi satu koleksi tunggal berdasarkan ID citra, lokasi, dan tanggal yang sama.

In [ ]:
# function for combine "Sentinel-2 SR" with "cloud probability"
def get_s2_sr_cld_col(aoi, start_date, end_date):
    # Import and filter S2 SR.
    s2_sr_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER)))

    # Import and filter s2cloudless.
    s2_cloudless_col = (ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
        .filterBounds(aoi)
        .filterDate(start_date, end_date))

    # Join the filtered s2cloudless collection to the SR collection by the 'system:index' property.
    return ee.ImageCollection(ee.Join.saveFirst('s2cloudless').apply(**{
        'primary': s2_sr_col,
        'secondary': s2_cloudless_col,
        'condition': ee.Filter.equals(**{
            'leftField': 'system:index',
            'rightField': 'system:index'
        })
    }))

In [ ]:
# Re-collect the S2 as we have changed our filtering params
s2_sr_cld_col = get_s2_sr_cld_col(AOI, START_DATE, END_DATE)

Function buat jadiin layer "probability" yang di s2cloudless kita buat di atas jadi masker biner (0 atau 1) berdasarkan CLD_PRB_THRESH yang di control variable

In [ ]:
def add_cloud_bands(img):
    # Get s2cloudless image, subset the probability band.
    cld_prb = ee.Image(img.get('s2cloudless')).select('probability')

    # Condition s2cloudless by the probability threshold value.
    is_cloud = cld_prb.gt(CLD_PRB_THRESH).rename('clouds')

    # Add the cloud probability layer and cloud mask as image bands.
    return img.addBands(ee.Image([cld_prb, is_cloud]))

Function buat bikin band baru (‘dark pixel’, ‘cloud transform', and ‘shadow’) ke dalam citra S2 SR

In [ ]:
def add_shadow_bands(img):
    # Identify water pixels from the SCL band.
    not_water = img.select('SCL').neq(6)

    # Identify dark NIR pixels that are not water (potential cloud shadow pixels).
    SR_BAND_SCALE = 1e4
    dark_pixels = img.select('B8').lt(NIR_DRK_THRESH*SR_BAND_SCALE).multiply(not_water).rename('dark_pixels')

    # Determine the direction to project cloud shadow from clouds (assumes UTM projection).
    shadow_azimuth = ee.Number(90).subtract(ee.Number(img.get('MEAN_SOLAR_AZIMUTH_ANGLE')));

    # Project shadows from clouds for the distance specified by the CLD_PRJ_DIST input.
    cld_proj = (img.select('clouds').directionalDistanceTransform(shadow_azimuth, CLD_PRJ_DIST*10)
        .reproject(**{'crs': img.select(0).projection(), 'scale': 100})
        .select('distance')
        .mask()
        .rename('cloud_transform'))

    # Identify the intersection of dark pixels with cloud shadow projection.
    shadows = cld_proj.multiply(dark_pixels).rename('shadows')

    # Add dark pixels, cloud projection, and identified shadows as image bands.
    return img.addBands(ee.Image([dark_pixels, cld_proj, shadows]))

Function buat assemble semua ('probability', 'cloud', 'dark pixel', 'cloud transform', 'shadow' ) dan produce final mask (pakai function add_cloud_bands dan shadow_bands)

In [ ]:
def add_cld_shdw_mask(img):
    # Add cloud component bands.
    img_cloud = add_cloud_bands(img)

    # Add cloud shadow component bands.
    img_cloud_shadow = add_shadow_bands(img_cloud)

    # Combine cloud and shadow mask, set cloud and shadow as value 1, else 0.
    is_cld_shdw = img_cloud_shadow.select('clouds').add(img_cloud_shadow.select('shadows')).gt(0)

    # Remove small cloud-shadow patches and dilate remaining pixels by BUFFER input.
    # 20 m scale is for speed, and assumes clouds don't require 10 m precision.
    is_cld_shdw = (is_cld_shdw.focalMin(2).focalMax(BUFFER*2/20)
        .reproject(**{'crs': img.select([0]).projection(), 'scale': 20})
        .rename('cloudmask'))

    # Add the final cloud-shadow mask to the image.
    return img_cloud_shadow.addBands(is_cld_shdw)

Function Apply cloud mask ke Sentinel-2 bands


In [ ]:
# Define a function to apply the cloud mask to the S2 spectral bands
def apply_cld_shdw_mask(img):
    cloudmask = img.select('cloudmask')
    # Mask cloudy pixels
    masked_img = img.updateMask(cloudmask.Not())
    # Keep cloudmask and probability bands
    cloudmask_band = img.select('cloudmask')
    probability_band = img.select('probability')
    masked_img = masked_img.addBands([cloudmask_band, probability_band], overwrite=True)
    return masked_img

In [ ]:
# Process the collection with .moasic as the compositing operator
s2_sr_mosaic = (s2_sr_cld_col.map(add_cld_shdw_mask)
                             .map(apply_cld_shdw_mask)
                             .mosaic()) #<- switched to mosaic to better preserve the cloud masking layers we want to now use for our k-means workflow

define_cloud_zones(img, edge_buffer=120) membagi citra jadi 3: Cloud Interior, Clear Interior, Cloud Edge

In [ ]:
def define_cloud_zones(img, edge_buffer=120):
    cloudmask = img.select('cloudmask')

    # Interior cloud pixels: cloudmask==1, away from edges (small erosion)
    cloud_interior = cloudmask.selfMask().focal_min(3)

    # Interior clear pixels: cloudmask==0, away from edges (medium erosion)
    clear_interior = cloudmask.Not().selfMask().focal_min(10)

    # Cloud edge pixels: buffer zone around clouds, but outside cloudmask
    cloud_edge = cloudmask.focal_max(edge_buffer / 10).And(cloudmask.Not())

    return cloud_interior, clear_interior, cloud_edge

multiple AOI points

In [ ]:
# Define AOIs for various land cover types.
aoi_points = {
    'urban_cbd':     ee.Geometry.Point(174.7633, -36.8485),  # downtown Auckland (Urban area)
    'forest':        ee.Geometry.Point(174.5243, -36.9365),  # Waitakere Ranges (bush)
    'farmland':      ee.Geometry.Point(174.5530, -36.8062),  # Taupaki
    'white_roof':     ee.Geometry.Point(174.8905, -36.9395),  # East Tamaki
}

sample_zones(img, AOI, edge_buffer=20) define wilayah sample zoning clouds

In [ ]:
def sample_zones(img, aoi_points, edge_buffer=20):
    cloudmask = img.select('cloudmask')

    cloud_interior = cloudmask.selfMask()
    clear_interior = cloudmask.Not().selfMask()
    cloud_edge = cloudmask.focal_max(edge_buffer).And(cloudmask.Not())

    sample_bands = ['B2', 'B3', 'B4', 'B5', 'B6' , 'B9', 'B8', 'B11', 'B12']
    if 'probability' in img.bandNames().getInfo():
        sample_bands.append('probability')

    img_for_sampling = img.select(sample_bands)

    cloud_samples = img_for_sampling.updateMask(cloud_interior).sample(
        region=aoi_points.buffer(2000), scale=10, numPixels=5000, seed=0, geometries=True)

    clear_samples = img_for_sampling.updateMask(clear_interior).sample(
        region=aoi_points.buffer(2000), scale=10, numPixels=5000, seed=1, geometries=True)

    edge_samples = img_for_sampling.updateMask(cloud_edge).sample(
        region=aoi_points.buffer(2000), scale=10, numPixels=5000, seed=2, geometries=True)

    return cloud_samples, clear_samples, edge_samples


def fc_to_df(fc) untuk convert EE feture class to a pandas df buat bikin grafik

In [ ]:
# Preparing an empty container list
cloud_dfs, clear_dfs, edge_dfs = [], [], []

  # Function to convert a earth engine feature class to a pandas df
def fc_to_df(fc):
    features = fc.getInfo()['features']
    rows = []
    for f in features:
        props = f['properties']
        coords = f['geometry']['coordinates']
        props['longitude'] = coords[0]
        props['latitude'] = coords[1]
        rows.append(props)
    return pd.DataFrame(rows)

# Sampling iteration per AOI location
for name, pt in aoi_points.items():
    # Gather our samples
    cloud_samples, clear_samples, edge_samples = sample_zones(s2_sr_mosaic, pt)
    # Append the DataFrame for each location to the container list.
    cloud_dfs.append(fc_to_df(cloud_samples))
    clear_dfs.append(fc_to_df(clear_samples))
    edge_dfs.append(fc_to_df(edge_samples))

# Carry out the conversion and sanity print out what we have
cloud_df = pd.concat(cloud_dfs, ignore_index=True)
clear_df = pd.concat(clear_dfs, ignore_index=True)
edge_df = pd.concat(edge_dfs, ignore_index=True)

print(f"Cloud samples: {len(cloud_df)}")
print(f"Clear samples: {len(clear_df)}")
print(f"Edge samples: {len(edge_df)}")
print("Cloud columns:", cloud_df.columns)

In [ ]:
# Histogram of cloud probability by class
plt.figure(figsize=(12, 5))
plt.hist(cloud_df['probability'], bins=30, histtype='step', label='Cloud')
plt.hist(clear_df['probability'], bins=30, histtype='step', label='Clear')
plt.hist(edge_df['probability'], bins=30, histtype='step', label='Edge')
plt.xlabel('Cloud Probability')
plt.ylabel('Frequency')
plt.legend()
plt.title('Distribution of Cloud Probability')
plt.show()

Pair plots band

In [ ]:
# Plot out all the bands, by the two 'labelled' classes that we want to discriminate between
# Add class labels
cloud_df['class'] = 'Cloud'
clear_df['class'] = 'Clear'

# Combine to one df
combined_df = pd.concat([cloud_df, clear_df])

# We will carry out somne class balancing here
# Find smallest class count
min_count = combined_df['class'].value_counts().min()

# Downsample each class to min_count
balanced_df = combined_df.groupby('class').sample(n=min_count, random_state=42)

# Now plot with balanced_df
# This is going to make a big plot that you will have to scroll around! Pay particular attention to the histogram plots on the diagonal.
bands = ['B2', 'B3', 'B4', 'B5', 'B6' , 'B8', 'B9', 'B11', 'B12']
sns.pairplot(balanced_df, vars=bands, hue='class', plot_kws={'alpha':0.3, 's':15}, height=2.5)


## K-Means Classified

In [ ]:
# Use the balanced_df from before for cloud and clear pixels only
train_df = balanced_df[balanced_df['class'].isin(['Cloud', 'Clear'])]

# Features to use for clustering
features = ['B9', 'B11', 'B12']

# Prepare training data (cloud=1, clear=0)
X_train = train_df[features].values
y_train = (train_df['class'] == 'Cloud').astype(int).values

# Fit k-means with 2 clusters on training data
kmeans = KMeans(n_clusters=2, random_state=42).fit(X_train)

# Map clusters to classes by majority vote on training data
labels, counts = np.unique(kmeans.labels_[y_train==0], return_counts=True)
clear_cluster = labels[np.argmax(counts)]
labels, counts = np.unique(kmeans.labels_[y_train==1], return_counts=True)
cloud_cluster = labels[np.argmax(counts)]

# Prepare edge pixels data for prediction
edge_features = edge_df[features].values

# Predict clusters for edge pixels
edge_clusters = kmeans.predict(edge_features)

# Assign class labels to edge pixels based on cluster
edge_df['predicted_class'] = ['Cloud' if c == cloud_cluster else 'Clear' for c in edge_clusters]

# Quick summary
print(edge_df['predicted_class'].value_counts())

plot up the results

In [ ]:
# Count of predicted classes
counts = edge_df['predicted_class'].value_counts()
print(counts)

# Scatter plot of edge pixels B4 vs B3 colored by predicted class ### CHANGE THIS TO YOUR OTHER BANDS ###
plt.figure(figsize=(10, 6))
for label, color in zip(['Cloud', 'Clear'], ['purple', 'green']):
    subset = edge_df[edge_df['predicted_class'] == label]
    plt.scatter(subset['B9'], subset['B11'], label=label, alpha=0.5, s=10, c=color)

plt.xlabel('B9 (Red)')
plt.ylabel('B12 (Green)')
plt.title('Edge Pixels Classified by K-Means')
plt.legend()
plt.show()

Finally, we will map this back onto earth engine via the following workflow:

1. Export the classified edge pixels back to Earth Engine as a FeatureCollection
2. Rasterize the classified edge points into an image layer
3. Visualize refined cloud mask in Earth Engine (geemap):

def df_to_ee_fc(df) Export to Earth Engine FC:

In [ ]:
# Function to go the other way! This time from a dataframe to the feature collection
def df_to_ee_fc(df):
    features = []
    for _, row in df.iterrows():
        geom = ee.Geometry.Point([row['longitude'], row['latitude']])
        props = {k: float(row[k]) for k in ['B2', 'B3', 'B4', 'B8', 'probability']}
        props['predicted_class'] = row['predicted_class']
        features.append(ee.Feature(geom, props))
    return ee.FeatureCollection(features)

edge_fc = df_to_ee_fc(edge_df)

class_to_numeric(feature) Rasterize into image layer:

In [ ]:
# Map predicted class to numeric
def class_to_numeric(feature):
    cls = feature.get('predicted_class')
    return feature.set('cloud_refined', ee.Number(ee.Algorithms.If(ee.String(cls).equals('Cloud'), 1, 0)))

edge_fc_num = edge_fc.map(class_to_numeric)

# Rasterize: paint 'cloud_refined' on a blank image
refined_cloud_mask = edge_fc_num.reduceToImage(
    properties=['cloud_refined'],
    reducer=ee.Reducer.first())

# Optionally, you can combine this refined mask with your existing cloudmask for a final refined mask
final_mask = s2_sr_mosaic.select('cloudmask').max(refined_cloud_mask).rename('cloudmask_refined')


Visualize

In [ ]:
Map = geemap.Map(basemap='HYBRID')
Map.centerObject(AOI, 10)

# Show original cloudmask
Map.addLayer(s2_sr_mosaic.select('cloudmask'), {'palette': ['white', 'red']}, 'Original cloudmask')

# Show refined cloudmask
Map.addLayer(final_mask.selfMask(), {'palette': ['blue']}, 'Refined cloudmask')

Map

## Classify edge pixels across a whole satellite image using the model that you have trained here using SciKitLearn.

1. Extract cluster centres from the trained K-means model.**bold text**

In [ ]:
cluster_centers = kmeans.cluster_centers_
print("Cluster centres:\n", cluster_centers)
print("Cloud cluster index:", cloud_cluster, "| Clear cluster index:", clear_cluster)

2. Per-pixel classification function: manual replication of kmeans.predict()

In [ ]:
def classify_by_centroids_ee(img, features, centers, cloud_idx):
    dist_bands = []
    for center in centers:
        sq_diff_sum = ee.Image.constant(0)
        for f, val in zip(features, center):
            diff_sq = img.select(f).subtract(ee.Number(float(val))).pow(2)
            sq_diff_sum = sq_diff_sum.add(diff_sq)
        dist_bands.append(sq_diff_sum.sqrt())

    dist_stack = ee.Image.cat(dist_bands).toArray()
    nearest_cluster = dist_stack.multiply(-1).arrayArgmax().arrayGet([0])

    return nearest_cluster.eq(ee.Number(int(cloud_idx))).rename('kmeans_cloud')

3. Define the edge zone in a raster format (dense, covering the entire image).

In [ ]:
cloud_interior_img, clear_interior_img, cloud_edge_img = define_cloud_zones(
    s2_sr_mosaic, edge_buffer=20
)

4. Run the classifier on the entire image.

In [ ]:
nearest_centroid_class = classify_by_centroids_ee(
    s2_sr_mosaic, features, cluster_centers, cloud_cluster
)

not_water = s2_sr_mosaic.select('SCL').neq(6)

edge_reclassified_as_cloud = (nearest_centroid_class
                               .multiply(cloud_edge_img)
                               .multiply(not_water)
                               .rename('cloudmask'))

final_mask_edge_refined = (s2_sr_mosaic.select('cloudmask')
                                       .max(edge_reclassified_as_cloud)
                                       .rename('cloudmask_edge_refined'))

5. Visualisation of edge pixel classification results across the entire image.

In [ ]:
s2_sr_cld_col_eval = get_s2_sr_cld_col(AOI, START_DATE, END_DATE)

Map2 = geemap.Map(basemap='HYBRID')
Map2.centerObject(AOI, 10)

Map2.addLayer(s2_sr_cld_col_eval, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
              'S2 ')
Map2.addLayer(s2_sr_mosaic.select('cloudmask').selfMask(), {'palette': ['white', 'red']},
              'Original cloudmask (s2cloudless)')
Map2.addLayer(cloud_edge_img.selfMask(), {'palette': ['yellow']},
              'Edge zone (candidate pixels, whole image)')
Map2.addLayer(edge_reclassified_as_cloud.selfMask(), {'palette': ['orange']},
              'Edge pixels reclassified as cloud (K-means)')
Map2.addLayer(final_mask_edge_refined.selfMask(), {'palette': ['blue']},
              'Final refined cloudmask (whole image)')

Map2